# SR vs CRITIC

In [23]:
import wandb
run = wandb.init()
artifact = run.use_artifact('agential/ambignq/run-ep382fce-likely-bird-8_eval:v0', type='run_table')
artifact_dir = artifact.download()

wandb:   1 of 1 files downloaded.  


In [24]:
import json

p = "/Users/vincent/Desktop/agential/experiments/artifacts/run-ep382fce-likely-bird-8_eval:v0/likely-bird-8_eval.table.json"
with open(p, "r") as f:
    data = json.load(f)

print(len(data))

7


In [25]:
import pandas as pd

critic_df = pd.DataFrame(data['data'], columns=data['columns'])

critic_df.head()

,question,answer,predicted_answer,EM,fuzzy_EM,llm_judge_eval,precision,recall,f1
0,When did ben stone leave law and order?,"[{'answer': None, 'qaPairs': [{'answer': ['May...",1994,0,1,1,1.0,0.333333,0.5
1,Who is magic in blood in blood out?,"[{'answer': ['Victor Rivers'], 'qaPairs': None...","Magic is Miklo Velka’s cousin, played by Jesse...",0,0,0,0.0,0.000000,0.0
2,Number of participating countries in tour de f...,"[{'answer': ['32'], 'qaPairs': None, 'type': '...",32,1,1,1,1.0,1.000000,1.0
3,Movie with james caan and james earl jones?,"[{'answer': ['Gardens of Stone'], 'qaPairs': N...",The Killer Elite (1975),0,0,0,0.0,0.000000,0.0
4,Who won the match of asia cup 2018?,"[{'answer': None, 'qaPairs': [{'answer': ['Ind...",India,1,1,1,1.0,1.000000,1.0


In [26]:
import wandb
run = wandb.init()
artifact = run.use_artifact('agential/ambignq/run-dhwoicja-stoic-glitter-11_eval:v0', type='run_table')
artifact_dir = artifact.download()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb:   1 of 1 files downloaded.  


In [27]:
import json
import os

p = "/Users/vincent/Desktop/agential/experiments/artifacts/run-dhwoicja-stoic-glitter-11_eval:v0/stoic-glitter-11_eval.table.json"
with open(p, "r") as f:
    data = json.load(f)



self_refine_df = pd.DataFrame(data['data'], columns=data['columns'])

self_refine_df.head()

,question,answer,predicted_answer,EM,fuzzy_EM,llm_judge_eval,precision,recall,f1
0,When did ben stone leave law and order?,"[{'answer': None, 'qaPairs': [{'answer': ['May...",Ben Stone left Law & Order in 1994.,0,0,1,0.142857,0.333333,0.2
1,Who is magic in blood in blood out?,"[{'answer': ['Victor Rivers'], 'qaPairs': None...","Magic is a character in the film ""Blood In Blo...",0,0,1,0.000000,0.000000,0.0
2,Number of participating countries in tour de f...,"[{'answer': ['32'], 'qaPairs': None, 'type': '...",Approximately 30,0,0,0,0.000000,0.000000,0.0
3,Movie with james caan and james earl jones?,"[{'answer': ['Gardens of Stone'], 'qaPairs': N...",The Gambler (1974),0,0,0,0.000000,0.000000,0.0
4,Who won the match of asia cup 2018?,"[{'answer': None, 'qaPairs': [{'answer': ['Ind...",India,1,1,1,1.000000,1.000000,1.0


In [28]:
# merge two tables
merged_df = pd.merge(critic_df, self_refine_df, on='question', suffixes=('_critic', '_self_refine'))

# List 1: self_refine gets it right (EM=1) and critic gets it wrong (EM=0)
self_refine_wins = merged_df[(merged_df['EM_self_refine'] == 1) & (merged_df['EM_critic'] == 0)]

# List 2: both failed (both get EM=0)
both_failed = merged_df[(merged_df['EM_self_refine'] == 0) & (merged_df['EM_critic'] == 0)]

print(f"Self-refine wins: {len(self_refine_wins)} questions")
print(f"Both failed: {len(both_failed)} questions")

# Display the questions where self-refine wins
print("\nQuestions where self-refine wins:")
for _, row in self_refine_wins.iterrows():
    print(f"Question: {row['question']}")
    print(f"Self-refine answer: {row['predicted_answer_self_refine']}")
    print(f"Critic answer: {row['predicted_answer_critic']}")
    print(f"Critic LLM Judge: {row['llm_judge_eval_critic']}")
    print(f"Self-refine LLM Judge: {row['llm_judge_eval_self_refine']}")
    print(f"Correct answer: {row['answer_critic']}")
    print("-" * 50)

Self-refine wins: 13 questions
Both failed: 149 questions

Questions where self-refine wins:
Question: When did new zealand win the americas cup?
Self-refine answer: 1995
Critic answer: 1995 (first win) or 2017 (one of the wins), depending on context.
Critic LLM Judge: 1
Self-refine LLM Judge: 1
Correct answer: [{'answer': None, 'qaPairs': [{'answer': ["29th America's Cup", '6–13 May 1995', '1995'], 'question': "When did the Royal New Zealand Yacht Squadron win its first America's Cup?"}, {'answer': ['20 February – 2 March 2000', "30th America's Cup", '2000'], 'question': "When did the Royal New Zealand Yacht Squadron win its first defense of the America's Cup?"}, {'answer': ["35th staging of the America's Cup yacht race", '17–26 June 2017', "2017 America's Cup", '2017'], 'question': "When did the Royal New Zealand Yacht Squadron win its second America's Cup?"}], 'type': 'multipleQAs'}]
--------------------------------------------------
Question: What is the lowest # on the fm dial?
Se

In [29]:
# Display the questions where both failed
print("\nQuestions where both failed:")
for _, row in both_failed.iterrows():
    print(f"Question: {row['question']}")
    print(f"Self-refine answer: {row['predicted_answer_self_refine']}")
    print(f"Critic answer: {row['predicted_answer_critic']}")
    print(f"Correct answer: {row['answer_critic']}")
    print("-" * 50)


Questions where both failed:
Question: When did ben stone leave law and order?
Self-refine answer: Ben Stone left Law & Order in 1994.
Critic answer: 1994
Correct answer: [{'answer': None, 'qaPairs': [{'answer': ['May 25, 1994'], 'question': 'What date did Ben Stone leave Law and Order?'}, {'answer': ['Season 4 episode 22'], 'question': 'What episode of Law and Order did Ben Stone last appear in?'}], 'type': 'multipleQAs'}]
--------------------------------------------------
Question: Who is magic in blood in blood out?
Self-refine answer: Magic is a character in the film "Blood In Blood Out," portrayed by actor Jesse Borrego.
Critic answer: Magic is Miklo Velka’s cousin, played by Jesse Borrego.
Correct answer: [{'answer': ['Victor Rivers'], 'qaPairs': None, 'type': 'singleAnswer'}]
--------------------------------------------------
Question: Movie with james caan and james earl jones?
Self-refine answer: The Gambler (1974)
Critic answer: The Killer Elite (1975)
Correct answer: [{'ans

In [30]:
critic_df.to_csv("critic.csv", index=False)

# Reflexion vs ReAct

In [31]:
import wandb
run = wandb.init()
artifact = run.use_artifact('agential/mbpp/run-n3cas241-hearty-hill-10_eval:v0', type='run_table')
artifact_dir = artifact.download()

wandb:   1 of 1 files downloaded.  


In [33]:
p = "/Users/vincent/Desktop/agential/experiments/artifacts/run-n3cas241-hearty-hill-10_eval:v0/hearty-hill-10_eval.table.json"
import json
with open(p, "r") as f:
    data = json.load(f)

reflexion_df = pd.DataFrame(data['data'], columns=data['columns'])


In [34]:
import wandb
run = wandb.init()
artifact = run.use_artifact('agential/mbpp/run-98fosw8u-fine-aardvark-6_eval:v0', type='run_table')
artifact_dir = artifact.download()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb:   1 of 1 files downloaded.  


In [35]:
p = "/Users/vincent/Desktop/agential/experiments/artifacts/run-98fosw8u-fine-aardvark-6_eval:v0/fine-aardvark-6_eval.table.json"
import json
with open(p, "r") as f:
    data = json.load(f)

react_df = pd.DataFrame(data['data'], columns=data['columns'])


In [38]:
sum(react_df.EM)/len(react_df)

0.92

In [39]:
sum(reflexion_df.EM)/len(reflexion_df)

0.91

In [42]:
# merge two tables
merged_df = pd.merge(react_df, reflexion_df, on='question', suffixes=('_react', '_reflexion'))

# List 1: self_refine gets it right (EM=1) and critic gets it wrong (EM=0)
react_wins = merged_df[(merged_df['EM_react'] == 1) & (merged_df['EM_reflexion'] == 0)]


print(f"ReAct wins: {len(react_wins)} questions")

# Display the questions where self-refine wins
print("\nQuestions where react wins:")
for _, row in react_wins.iterrows():
    print(f"Question: {row['question']}")
    print(f"react answer: {row['predicted_answer_react']}")
    print(f"reflexion answer: {row['predicted_answer_reflexion']}")
    print(f"Correct answer: {row['answer_reflexion']}")
    print("-" * 50)

ReAct wins: 4 questions

Questions where react wins:
Question: Write a function to get the angle of a complex number.
react answer: ```python
import cmath
def angle_complex(real, imag):
    z = complex(real, imag.imag if isinstance(imag, complex) else imag)
    return cmath.phase(z)
```
reflexion answer: ```python
import cmath
def angle_complex(z):
    return cmath.phase(z)
```
Correct answer: import math

assert math.isclose(angle_complex(0,1j), 1.5707963267948966, rel_tol=0.001)
assert math.isclose(angle_complex(2,1j), 0.4636476090008061, rel_tol=0.001)
assert math.isclose(angle_complex(0,2j), 1.5707963267948966, rel_tol=0.001)
--------------------------------------------------
Question: Write a function to find the first adverb and their positions in a given sentence.
react answer: ```python
import re
def find_adverb_position(sentence):
    # Use regex to find the first word (sequence of letters) at the start of the sentence
    match = re.match(r'([a-zA-Z]+)', sentence)
    if matc

---

In [43]:
import wandb
run = wandb.init()
artifact = run.use_artifact('agential/fever/run-stzyrj9q-twilight-smoke-6_eval:v0', type='run_table')
artifact_dir = artifact.download()

wandb:   1 of 1 files downloaded.  


In [45]:
p = "/Users/vincent/Desktop/agential/experiments/artifacts/run-stzyrj9q-twilight-smoke-6_eval:v0/twilight-smoke-6_eval.table.json"
import json
with open(p, "r") as f:
    data = json.load(f)

react_df = pd.DataFrame(data['data'], columns=data['columns'])


In [ ]:
import wandb
run = wandb.init()
artifact = run.use_artifact('agential/fever/run-srjw00yj-swift-vortex-9_eval:v0', type='run_table')
artifact_dir = artifact.download()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb:   1 of 1 files downloaded.  


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [47]:
p = "/Users/vincent/Desktop/agential/experiments/artifacts/run-srjw00yj-swift-vortex-9_eval:v0/swift-vortex-9_eval.table.json"
import json
with open(p, "r") as f:
    data = json.load(f)

reflexion_df = pd.DataFrame(data['data'], columns=data['columns'])


In [49]:
sum(reflexion_df.EM)/len(reflexion_df)

0.385

In [50]:
# merge two tables
merged_df = pd.merge(react_df, reflexion_df, on='question', suffixes=('_react', '_reflexion'))

# List 1: self_refine gets it right (EM=1) and critic gets it wrong (EM=0)
react_wins = merged_df[(merged_df['EM_react'] == 1) & (merged_df['EM_reflexion'] == 0)]


print(f"ReAct wins: {len(react_wins)} questions")

# Display the questions where self-refine wins
print("\nQuestions where react wins:")
for _, row in react_wins.iterrows():
    print(f"Question: {row['question']}")
    print(f"react answer: {row['predicted_answer_react']}")
    print(f"reflexion answer: {row['predicted_answer_reflexion']}")
    print(f"Correct answer: {row['answer_reflexion']}")
    print("-" * 50)

ReAct wins: 24 questions

Questions where react wins:
Question: Ann Richards was a Muslim.
react answer: NOT ENOUGH INFO
reflexion answer: REFUTES
Correct answer: NOT ENOUGH INFO
--------------------------------------------------
Question: Gray Matter Interactive Studios, Inc. was acquired by Activision in January 2002.
react answer: SUPPORTS
reflexion answer: NOT ENOUGH INFORMATION
Correct answer: SUPPORTS
--------------------------------------------------
Question: Harold Macmillan died on Monday December 29, 1986.
react answer: SUPPORTS
reflexion answer: NOT ENOUGH INFO
Correct answer: SUPPORTS
--------------------------------------------------
Question: Omar Khadr was sentenced to two years in prison.
react answer: NOT ENOUGH INFO
reflexion answer: REFUTES
Correct answer: NOT ENOUGH INFO
--------------------------------------------------
Question: Leonard Nimoy is a husband.
react answer: NOT ENOUGH INFO
reflexion answer: SUPPORTS
Correct answer: NOT ENOUGH INFO
-------------------

In [62]:
react_wins.head()

,question,answer_react,predicted_answer_react,EM_react,fuzzy_EM_react,llm_judge_eval_react,precision_react,recall_react,f1_react,answer_reflexion,predicted_answer_reflexion,EM_reflexion,fuzzy_EM_reflexion,llm_judge_eval_reflexion,precision_reflexion,recall_reflexion,f1_reflexion
6,Ann Richards was a Muslim.,NOT ENOUGH INFO,NOT ENOUGH INFO,1,1,0,1.0,1.0,1.0,NOT ENOUGH INFO,REFUTES,0,0,0,0.0,0.0,0.0
33,"Gray Matter Interactive Studios, Inc. was acqu...",SUPPORTS,SUPPORTS,1,1,0,1.0,1.0,1.0,SUPPORTS,NOT ENOUGH INFORMATION,0,0,0,0.0,0.0,0.0
42,"Harold Macmillan died on Monday December 29, 1...",SUPPORTS,SUPPORTS,1,1,0,1.0,1.0,1.0,SUPPORTS,NOT ENOUGH INFO,0,0,0,0.0,0.0,0.0
48,Omar Khadr was sentenced to two years in prison.,NOT ENOUGH INFO,NOT ENOUGH INFO,1,1,0,1.0,1.0,1.0,NOT ENOUGH INFO,REFUTES,0,0,0,0.0,0.0,0.0
64,Leonard Nimoy is a husband.,NOT ENOUGH INFO,NOT ENOUGH INFO,1,1,0,1.0,1.0,1.0,NOT ENOUGH INFO,SUPPORTS,0,0,0,0.0,0.0,0.0


In [60]:
react_wins.answer_react.values.tolist()

['NOT ENOUGH INFO',
 'SUPPORTS',
 'SUPPORTS',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'REFUTES',
 'NOT ENOUGH INFO',
 'SUPPORTS',
 'SUPPORTS',
 'NOT ENOUGH INFO',
 'SUPPORTS',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'SUPPORTS',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO',
 'NOT ENOUGH INFO']

In [66]:
a = """
It appears this Ann Richards is an Australian   
actress, not the American politician who is     
more likely the subject of the claim. There is  
NOT ENOUGH INFORMATION here to determine her    
religion, and this is likely the wrong person.  

Action 3: Finish[NOT ENOUGH INFORMATION] 
"""

def parse_thought(text: str) -> str:
    """
    Parse thought from text, handling various formats.

    Handles:
    - Thought 1: <content>
    - Thought: <content>
    - <content> (no prefix)
    - Multi-line thoughts (splits at Action/Observation)

    Args:
        text (str): Raw thought text

    Returns:
        str: Cleaned thought content
    """
    text = text.strip()

    # Remove Thought X: prefix if present using string operations
    if ":" in text:
        # Split by first colon and take the part after it
        parts = text.split(":", 1)
        if len(parts) > 1:
            text = parts[0].strip()
    
    # Split at Action and take the first part
    return text.split("Action")[0].strip()

parse_thought(a)

'It appears this Ann Richards is an Australian   \nactress, not the American politician who is     \nmore likely the subject of the claim. There is  \nNOT ENOUGH INFORMATION here to determine her    \nreligion, and this is likely the wrong person.'